# Task 1.1 — Core Contribution / Architecture of CS4VM
**Paper:** Cost-Sensitive Semi-Supervised Support Vector Machine (CS4VM)  
**Authors:** Li, Kwok, Zhou — AAAI 2010


## Step-by-Step Method Description

### Step 1: Problem Setup — Labeled + Unlabeled Data with Unequal Costs
- **What happens:** We are given a small set of labeled examples {(x₁,y₁),…,(xₗ,yₗ)} and a larger set of unlabeled examples {xₗ₊₁,…,xₗ₊ᵤ}. Labels y ∈ {+1, −1}. The cost of misclassifying a positive (negative) instance is c(+1) (c(−1)), and these costs are unequal.
- **Reference:** Section "CS4VM — Formulation", Eq. (1)–(2)
- **Purpose:** This dual challenge — few labels and unequal misclassification costs — is the core motivation. Existing cost-sensitive methods ignore unlabeled data; semi-supervised methods ignore costs. CS4VM addresses both simultaneously.

### Step 2: CS4VM Formulation — Joint Optimization over Labels and Classifier
- **What happens:** Because labels of the unlabeled data are unknown, the CS4VM (Eq. 2) simultaneously optimizes over both the decision function f and the predicted labels ŷᵢ for unlabeled instances. A **balance constraint** ∑ŷᵢ = r prevents the degenerate solution that assigns all unlabeled points to one class.
- **Reference:** Eq. (2)–(3); balance constraint defined with user-specified r
- **Purpose:** Encodes the intuition from S3VM (semi-supervised SVM) but with asymmetric, cost-weighted hinge losses. When c(+1) ≠ c(−1) the loss function (Figure 1) is no longer symmetric.

### Step 3: Label-Mean Reformulation — Reducing Labels to a Single Statistic
- **What happens:** Instead of optimizing over all 2ᵘ discrete label assignments, the paper shows (Section "Label Means for CS4VM", Eq. 5) that the CS4VM objective depends on the unlabeled data **only through** their class-conditional means in feature space: m₊ = (1/u₊)∑_{y*ᵢ=+1} φ(xᵢ) and m₋ = (1/u₋)∑_{y*ᵢ=−1} φ(xᵢ).
- **Reference:** Eq. (5)–(6); this leads to the simplified problem Eq. (6)
- **Purpose:** This is the mathematical core of the paper. The label means decouple the cost weights from the discrete label signs, converting an intractable combinatorial problem into a convex QP (Eq. 9).

### Step 4: Label Mean Estimation — Iterative Alternating Optimization
- **What happens:** To estimate label means, the algorithm alternates (Eq. 7–8):  
  (a) Fix the binary assignment vector d ∈ {0,1}ᵘ and solve for the maximum-margin (w, b, ρ) via standard SVM training.  
  (b) Fix (w, b, ρ) and solve for d via a linear program that assigns u₊ unlabeled points as positive (maximising the separating margin between class means).
- **Reference:** Section "Estimating Label Means", Eq. (7)–(8); Δ = {d | dᵢ ∈ {0,1}, d'1 = u₊}
- **Purpose:** Efficiently estimates which unlabeled instances belong to each class, without the exponential-time exhaustive search over all label assignments.

### Step 5: Plug-In Label Means — Solve Final Convex QP
- **What happens:** The estimated m̂₊ and m̂₋ from Step 4 are substituted into Eq. (6). Because the constraints become linear and the objective is convex, the resulting problem is solved as a standard **quadratic program** (Eq. 9 — the dual) using an efficient SMO-based SVM solver (e.g., LIBSVM).
- **Reference:** Eq. (9); Section "Solving Eq. 6 with Estimated Label Means"
- **Purpose:** Achieves the same computational efficiency as a supervised SVM despite leveraging unlabeled data.

### Step 6: Prediction
- **What happens:** For a new test point x, the learned f(x) = w'φ(x) + b classifies it as +1 if f(x) ≥ 0 and −1 otherwise.
- **Reference:** Standard SVM prediction using w from Eq. (9)
- **Purpose:** Produces cost-aware, semi-supervised predictions.

---

## Final Summary
CS4VM solves the problem of binary classification where labeled data is scarce and misclassification costs are unequal. The authors' key claim is that representing the unlabeled data by their **class-conditional label means** — a scalar statistic rather than a full label vector — is sufficient to approximate the supervised cost-sensitive SVM that has access to all ground-truth labels, enabling a fast two-step algorithm that outperforms both cost-insensitive semi-supervised methods (TSVM, Laplacian SVM) and supervised cost-sensitive methods.
